# 멀티모달 RAG — 논문 PDF의 텍스트·표·그림으로 답하기

| 항목 | 내용 |
|---|---|
| 분석 PDF | `data/invest.pdf` — Song & Ermon, *Generative Modeling by Estimating Gradients of the Data Distribution* (arXiv:1907.05600, 23쪽) |
| 준비물 | OpenAI API 키(`.env`), Poppler, Tesseract |
| 결과물 | 한국어로 질문하면 논문의 텍스트·표·그림을 찾아 근거와 함께 답하는 RAG |

**사용법**: 위에서부터 셀을 하나씩 `Shift + Enter`로 실행하고, **출력이 남은 상태로 저장**한 뒤 커밋합니다.

## 목차

| 파트 | 내용 | 셀 |
|---|---|---|
| 0 | 전체 그림 | 개념 |
| 1 | 환경 준비 (임포트·API 키·설정값) | [1]~[3] |
| 2 | 1단계 추출: 텍스트 / 표 / 그림 | [4]~[6] |
| 3 | 2단계 요약: 텍스트·표·그림 요약 | [7]~[8] |
| 4 | 3단계 검색기: 요약 검색 → 원본 반환 | [9] |
| 5 | LangChain 체인: 번역 체인 / RAG 체인 | [10]~[11] |
| 6 | 전체 파이프라인 & 질의응답 5건 | [12]~[14] |

---
# Part 0. 전체 그림

### 사전 준비 단계 (최초 1회)
```
 data/invest.pdf ──[1단계] partition_pdf(hi_res)──▶ 텍스트 블록 / 표(HTML) / 그림(data/figures/*.jpg)
                                                       │
                                     [2단계] 요약 (gpt-4o-mini, 그림은 비전 입력)
                                                       ▼
                    요약 ──▶ Chroma (검색용)      원본 ──▶ InMemoryStore (답변용)      ← doc_id 로 연결
```

### 질문할 때
```
 사용자(한국어) "Table 1에서 NCSN보다 점수가 높은 모델은?"
        │  ① 번역 체인                         ← LangChain LCEL
        ▼
 "Which model scores higher than NCSN in Table 1?"
        │  ② 요약 임베딩 검색 → doc_id 로 원본 꺼내기     ← Retrieval
        ▼
 [table p.7] <table>...</table>,  [image p.4] (base64) ...
        │  ③ 원본 텍스트·표 + 그림(data URL)으로 프롬프트 구성  ← Augmentation
        │  ④ RAG 체인 (gpt-4o-mini 비전)                   ← Generation
        │  ⑤ 번역 체인
        ▼
 한국어 답변 + 근거(종류·페이지) + 검색된 그림
```

### 왜 요약으로 검색하고 원본으로 생성하나?
- 표 HTML은 태그·숫자 위주라 그대로 임베딩하면 의미가 잘 안 잡히고, 그림은 텍스트 임베딩을 할 수 없습니다 → **요약(문장)으로 검색**
- 요약은 수치·세부가 빠질 수 있습니다 → **답변은 원본(표 전체, 그림 자체)으로 생성**

---
# Part 1. 환경 준비

In [ ]:
# ============================================================
# [1] 라이브러리 임포트 & 환경 점검
# ============================================================
import os
import shutil
import sys
import time
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv

# 노트북은 notebooks/ 에서 실행되므로 프로젝트 루트로 이동해 src/ 를 임포트할 수 있게 합니다.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- LangChain ---
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

# --- 이미지 처리 & 노트북 출력 ---
from IPython.display import Markdown, display
from PIL import Image

# --- 프로젝트 모듈 (src/) ---
from src.extract import (categorize_elements, extract_pdf_elements, list_figures,
                         load_pickle, save_pickle)
from src.retriever import create_multi_vector_retriever
from src.summarize import summarize_elements, summarize_images

print("Python   :", sys.version.split()[0], "/", sys.prefix)
print("Poppler  :", "OK" if shutil.which("pdftoppm") else "✗ 미설치 — Path 확인 후 재시작")
print("Tesseract:", "OK" if shutil.which("tesseract") else "✗ 미설치 — Path 확인 후 재시작")
for mod in ["unstructured", "unstructured_inference", "langchain_classic", "chromadb", "tiktoken"]:
    try:
        __import__(mod)
        print(f"{mod:22s}: OK")
    except ImportError:
        print(f"{mod:22s}: ✗ 미설치")

In [ ]:
# ============================================================
# [2] OpenAI API 키 설정
# ============================================================
load_dotenv()                                   # .env 파일을 찾아 환경변수로 등록
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")    # 환경변수에서 키 읽기 (없으면 None)

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY 가 설정되지 않았습니다. .env 파일을 확인하세요.")

# 키 전체를 출력하면 보안상 위험하므로 앞 4자리만 확인용으로 출력
print("API 키 설정 완료:", OPENAI_API_KEY[:4] + "****")

In [ ]:
# ============================================================
# [3] 설정값 (한 곳에서 관리)
# ============================================================
PDF_PATH      = PROJECT_ROOT / "data" / "invest.pdf"
FIGURES_DIR   = PROJECT_ROOT / "data" / "figures"      # 추출된 그림 저장 위치
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"    # 추출·요약 결과 저장 (Git 제외)

SUMMARY_MODEL     = "gpt-4o-mini"   # 텍스트·표·그림 요약 (비전 입력 지원)
RAG_MODEL         = "gpt-4o-mini"   # 원본 텍스트·표·그림을 보고 답변
TRANSLATION_MODEL = "gpt-4o-mini"   # 한↔영 번역
N_RESULTS         = 4               # 질문 하나당 검색할 원본 수

REBUILD = False   # True 로 바꾸면 저장본을 무시하고 추출·요약을 처음부터 다시 실행

print("PDF 존재 여부:", PDF_PATH.exists(), PDF_PATH)
print("그림 폴더    :", FIGURES_DIR)

> **나의 메모** : (이 칸을 더블클릭해서 알게 된 점을 적어 보세요)
>
> - 

---
# Part 2. 1단계 추출 — 텍스트 · 표 · 그림 (수업 01)

| 옵션 | 값 | 빼면 생기는 일 |
| --- | --- | --- |
| `strategy` | `"hi_res"` | 표·그림 영역을 인식하지 못함 |
| `extract_images_in_pdf` | `True` | 그림이 통째로 사라짐 |
| `infer_table_structure` | `True` | Table 1이 줄글로 뭉개짐 |
| `chunking_strategy` | `"by_title"` (4000 / 3800 / 2000) | 섹션 의미가 끊긴 블록이 생김 |

> ⏱️ hi_res는 몇 분 걸립니다. 결과는 `data/processed/extracted.pkl`에 저장되고, 다음부터는 저장본을 불러옵니다.

In [ ]:
# ============================================================
# [4] 추출 실행 (저장본이 있으면 불러옴)
# ============================================================
EXTRACTED_PKL = PROCESSED_DIR / "extracted.pkl"

if EXTRACTED_PKL.exists() and not REBUILD:
    data = load_pickle(EXTRACTED_PKL)
    print("저장본 불러옴:", EXTRACTED_PKL)
else:
    start = time.time()
    raw_pdf_elements = extract_pdf_elements(PDF_PATH, FIGURES_DIR)
    texts, tables = categorize_elements(raw_pdf_elements)
    data = {
        "texts": texts,
        "tables": tables,
        "images": list_figures(FIGURES_DIR),
        "type_counts": Counter(str(type(el)).split(".")[-1].strip("'>") for el in raw_pdf_elements),
    }
    save_pickle(data, EXTRACTED_PKL)
    print(f"추출 완료 — {time.time() - start:.1f}초 소요")

texts, tables, images = data["texts"], data["tables"], data["images"]

print("요소 타입 분포 :", dict(data["type_counts"]))
print("텍스트 블록 수 :", len(texts))
print("표 블록 수     :", len(tables))
print("그림 수        :", len(images), [Path(i["image_path"]).name for i in images])

In [ ]:
# ============================================================
# [5] 추출 결과 확인 — 텍스트 / 표
# ============================================================
print("=" * 60)
print(f"[텍스트 p.{texts[0]['page']}] 앞 300자")
print("=" * 60)
print(texts[0]["content"][:300], "\n")

for t in tables:
    print("=" * 60)
    print(f"[표 p.{t['page']}] HTML 앞 600자")
    print("=" * 60)
    print(t["content"][:600], "\n")

In [ ]:
# ============================================================
# [6] 추출 결과 확인 — 그림 (FashionRAG show_image_grid)
# ============================================================
def show_image_grid(paths, thumb=220):
    """여러 이미지를 가로로 이어 붙인 '썸네일 한 장'으로 출력합니다."""
    imgs = [Image.open(p).convert("RGB") for p in paths]
    for im in imgs:
        im.thumbnail((thumb, thumb))
    # 가로로 나란히 붙일 빈 캔버스 생성 (흰 배경, 이미지 사이 10px 여백)
    canvas = Image.new("RGB", (sum(im.width for im in imgs) + 10 * (len(imgs) - 1),
                               max(im.height for im in imgs)), "white")
    x = 0
    for im in imgs:
        canvas.paste(im, (x, 0))
        x += im.width + 10
    display(canvas)


print([f"p.{i['page']}" for i in images[:6]])
show_image_grid([i["image_path"] for i in images[:6]])

> **나의 메모** : (이 칸을 더블클릭해서 알게 된 점을 적어 보세요)
>
> - 

---
# Part 3. 2단계 요약 — 검색용 색인 만들기

- 텍스트·표: `ChatPromptTemplate | ChatOpenAI | StrOutputParser` (LCEL)
- 그림: 같은 LCEL 체인의 user 메시지에 `data:image/jpeg;base64,...` 이미지 블록을 넣어 비전 모델로 요약
- 논문이 영어이므로 **요약 프롬프트와 요약도 영어**로 만듭니다 (질문은 Part 5의 번역 체인으로 영어로 바꿈)

> 결과는 `data/processed/summaries.pkl`에 저장해 API 비용을 다시 쓰지 않게 합니다.

In [ ]:
# ============================================================
# [7] 요약 실행 (저장본이 있으면 불러옴)
# ============================================================
SUMMARY_PKL = PROCESSED_DIR / "summaries.pkl"

if SUMMARY_PKL.exists() and not REBUILD:
    summaries = load_pickle(SUMMARY_PKL)
    print("저장본 불러옴:", SUMMARY_PKL)
else:
    start = time.time()
    summaries = {
        "texts":  summarize_elements(texts,  model=SUMMARY_MODEL),
        "tables": summarize_elements(tables, model=SUMMARY_MODEL),
        "images": summarize_images(images,   model=SUMMARY_MODEL),
    }
    save_pickle(summaries, SUMMARY_PKL)
    print(f"요약 완료 — {time.time() - start:.1f}초 소요")

text_summaries, table_summaries, image_summaries = summaries["texts"], summaries["tables"], summaries["images"]
print(f"텍스트 요약 {len(text_summaries)}개 / 표 요약 {len(table_summaries)}개 / 그림 요약 {len(image_summaries)}개")

In [ ]:
# ============================================================
# [8] 요약 결과 확인
# ============================================================
print(f"[텍스트 요약 p.{texts[0]['page']}]\n{text_summaries[0]}\n")
for t, s in zip(tables, table_summaries):
    print(f"[표 요약 p.{t['page']}]\n{s}\n")
for img, s in list(zip(images, image_summaries))[:4]:
    print(f"[그림 요약 {Path(img['image_path']).name}]\n{s}\n")

> **나의 메모** : (이 칸을 더블클릭해서 알게 된 점을 적어 보세요)
>
> - 

---
# Part 4. 3단계 검색기 — 요약으로 찾고 원본을 돌려받기 (rag-parser)

| 저장소 | 넣는 것 | 역할 |
|---|---|---|
| `vectorstore` (Chroma) | 요약 임베딩 | 검색 대상 |
| `docstore` (InMemoryStore) | 원본: 텍스트, 표 HTML, 그림 base64 | 답변 재료 |

`MultiVectorRetriever`가 요약 검색 결과의 `doc_id`로 원본을 꺼내 돌려줍니다.

In [ ]:
# ============================================================
# [9] 다중 벡터 검색기 생성 & 검색 테스트
# ============================================================
mm_retriever = create_multi_vector_retriever(
    [(texts, text_summaries), (tables, table_summaries), (images, image_summaries)],
    k=N_RESULTS,
)

# '요약'으로 찾았지만 돌려받는 것은 '원본'입니다.
docs = mm_retriever.invoke("Inception and FID scores of NCSN on CIFAR-10")
for d in docs:
    kind, page = d.metadata["kind"], d.metadata["page"]
    preview = "(그림 base64 원본)" if kind == "image" else d.page_content[:120].replace("\n", " ")
    print(f"[{kind} p.{page}] {preview}")

> **나의 메모** : (이 칸을 더블클릭해서 알게 된 점을 적어 보세요)
>
> - 

---
# Part 5. LangChain 체인 — 번역 체인 & RAG 체인 (FashionRAG)

```
 {"text": ..., "target_lang": ...}  ─ ChatPromptTemplate ─▶ [System, Human] ─ ChatOpenAI ─▶ AIMessage ─ StrOutputParser ─▶ str
```

In [ ]:
# ============================================================
# [10] 번역 체인 (LCEL: 프롬프트 | 모델 | 파서)
# ============================================================
# temperature=0.0 : 번역처럼 '정답에 가까운 출력'이 필요할 때 무작위성 최소화
translation_model = ChatOpenAI(model=TRANSLATION_MODEL, temperature=0.0)

translation_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the following text into {target_lang}. "
               "Keep technical terms, model names and numbers as they are. "
               "Output only the translation."),
    ("user", "{text}"),
])

translation_chain = translation_prompt | translation_model | StrOutputParser()


def translate(text, target_lang):
    """text 를 target_lang(예: 'English', 'Korean')으로 번역한 문자열을 반환합니다."""
    return translation_chain.invoke({"text": text, "target_lang": target_lang})


print(translate("Table 1에서 NCSN의 FID는 얼마인가요?", "English"))

In [ ]:
# ============================================================
# [11] RAG 체인: 원본 텍스트·표 + 그림을 함께 LLM 에 전달
# ============================================================
def setup_rag_chain():
    """논문 질의응답용 '프롬프트 | 비전모델 | 파서' 체인을 반환합니다."""
    rag_model = ChatOpenAI(model=RAG_MODEL, temperature=0.0)

    # system : 모델의 역할 — 제공된 근거로만 답하고, 근거가 없으면 없다고 말하기
    # user   : 질문 + 원본 텍스트·표(context)
    # image_messages : 검색된 그림을 data URL 이미지 블록으로 담은 메시지 (없으면 비어 있음)
    rag_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are an assistant answering questions about a machine learning research paper. "
         "Answer only from the provided context: text passages, tables (HTML) and figures. "
         "Keep numbers and model names exactly as written. "
         "If the context is not enough, say \"The provided context does not contain the answer.\""),
        ("user", "Question: {user_query}\n\nContext:\n{context}"),
        MessagesPlaceholder("image_messages", optional=True),
    ])
    return rag_prompt | rag_model | StrOutputParser()


def format_prompt_inputs(docs, user_query):
    """검색된 원본 문서와 질문으로 RAG 프롬프트 입력 dict 를 만듭니다."""
    context_parts, image_blocks = [], []
    for d in docs:
        kind, page = d.metadata["kind"], d.metadata.get("page")
        tag = f"[{kind} p.{page}]" if page else f"[{kind}]"
        if kind == "image":
            context_parts.append(f"{tag} (see attached figure)")
            image_blocks.append({"type": "image_url",
                                 "image_url": {"url": f"data:image/jpeg;base64,{d.page_content}"}})
        else:
            context_parts.append(f"{tag}\n{d.page_content}")

    return {
        "user_query": user_query,
        "context": "\n\n".join(context_parts),
        "image_messages": [HumanMessage(content=image_blocks)] if image_blocks else [],
    }


def sources(docs):
    """'table p.7, image p.4' 형태로 근거 원본의 종류와 페이지를 적습니다."""
    return ", ".join(f"{d.metadata['kind']} p.{d.metadata['page']}" if d.metadata.get("page")
                     else d.metadata["kind"] for d in docs)


rag_chain = setup_rag_chain()
print("RAG 체인 준비 완료")

> **나의 메모** : (이 칸을 더블클릭해서 알게 된 점을 적어 보세요)
>
> - 

---
# Part 6. 전체 파이프라인 & 질의응답 5건

| 단계 | 함수 | RAG 역할 |
|---|---|---|
| ① 질문 번역 | `translate(query_ko, "English")` | 전처리 |
| ② 검색 | `retriever.invoke(query_en)` | **R**etrieval |
| ③ 프롬프트 입력 구성 | `format_prompt_inputs(docs, query_en)` | **A**ugmentation |
| ④ 답변 생성 | `rag_chain.invoke(prompt_input)` | **G**eneration |
| ⑤ 답변 번역 | `translate(answer_en, "Korean")` | 후처리 |

In [ ]:
# ============================================================
# [12] 전체 파이프라인: 한국어 질문 -> 번역 -> 검색 -> RAG 체인 -> 번역 -> 출력
# ============================================================
def ask_paper_rag(query_ko, retriever, show=True):
    """한국어 질문 하나를 받아 전체 과정을 실행하고 결과 dict 를 반환합니다."""
    query_en = translate(query_ko, "English")                 # ①
    docs = retriever.invoke(query_en)                         # ②
    prompt_input = format_prompt_inputs(docs, query_en)       # ③
    answer_en = rag_chain.invoke(prompt_input)                # ④
    answer_ko = translate(answer_en, "Korean")                # ⑤

    if show:
        print("질문        :", query_ko)
        print("번역된 질문 :", query_en)
        print("근거 원본   :", sources(docs))
        display(Markdown(answer_ko))
        image_paths = [d.metadata["image_path"] for d in docs if d.metadata["kind"] == "image"]
        if image_paths:
            show_image_grid(image_paths)

    return {"query_ko": query_ko, "query_en": query_en, "docs": docs,
            "answer_en": answer_en, "answer_ko": answer_ko, "sources": sources(docs)}

In [ ]:
# ============================================================
# [13] 질의응답 5건
# ============================================================
QUESTIONS = [
    "CIFAR-10 unconditional 설정에서 NCSN의 Inception score와 FID는 각각 얼마인가요?",
    "Table 1에서 Inception score가 NCSN보다 높은 모델은 무엇이고, 어떤 설정인가요?",
    "Figure 2에서 빨간 사각형으로 표시된 영역은 무엇을 의미하나요?",
    "Figure 3에서 annealed Langevin dynamics와 일반 Langevin dynamics의 결과는 어떻게 다른가요?",
    "이미지 생성 실험에서 노이즈 수준 σ는 몇 개, 어떤 범위로 설정했나요?",
]

mm_results = []
for i, q in enumerate(QUESTIONS, start=1):
    print(f"\n{'=' * 30} Q{i} {'=' * 30}")
    mm_results.append(ask_paper_rag(q, mm_retriever))

> **나의 메모** : (이 칸을 더블클릭해서 알게 된 점을 적어 보세요)
>
> - 

In [ ]:
# ============================================================
# [14] README 에 붙여 넣을 질의응답 표
# ============================================================
def one_line(s):
    return s.replace("\n", " ").replace("|", "\\|")

print("| # | 질문 | 답변 | 근거가 된 원본 종류 |")
print("| --- | --- | --- | --- |")
for i, r in enumerate(mm_results, start=1):
    kinds = ", ".join(sorted({d.metadata["kind"] for d in r["docs"]}))
    print(f"| {i} | {one_line(r['query_ko'])} | {one_line(r['answer_ko'])} | {kinds} ({r['sources']}) |")

> **나의 메모** : (이 칸을 더블클릭해서 알게 된 점을 적어 보세요)
>
> - 